# 29 — Vision-Language Adapters

In [ ]:
from pathlib import Path
import math, random, re, json, os, statistics, time, queue, threading
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except Exception as e:
    TfidfVectorizer = None
    cosine_similarity = None
    print('Optional sklearn import failed:', e)

# Make notebooks runnable from the pack root OR from a notebook subdirectory.
def find_pack_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for p in [start] + list(start.parents):
        if (p / 'data' / 'tiny_corpus.txt').exists():
            return p
    # Fallback for the sandbox path used to create this pack.
    sandbox = Path('/mnt/data/advanced_llm_systems_notebook_pack')
    if (sandbox / 'data' / 'tiny_corpus.txt').exists():
        return sandbox
    return start

ROOT = find_pack_root()
DATA = ROOT / 'data'
print('Pack root:', ROOT)
print('PyTorch:', torch.__version__)
torch.set_num_threads(1)
torch.manual_seed(7)
random.seed(7)
np.random.seed(7)

def basic_tokenize(text: str):
    return re.findall(r"[a-zA-Z]+|\d+|[^\w\s]", text.lower())

def count_params(model):
    return sum(p.numel() for p in model.parameters())

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        q,k,v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k = k.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        v = v.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        att = q @ k.transpose(-2,-1) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones(T,T,device=x.device)).bool()
        att = att.masked_fill(~mask[None,None,:,:], float('-inf'))
        w = torch.softmax(att, dim=-1)
        y = self.dropout(w) @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(y)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio*d_model), nn.GELU(),
            nn.Linear(mlp_ratio*d_model, d_model), nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

# Shared toy RAG utilities for notebooks that need retrieval.
def load_rag_docs():
    path = DATA / 'rag_docs.jsonl'
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

docs = load_rag_docs()
_retriever_state = {}
def build_retriever(docs_in=None):
    docs_in = docs if docs_in is None else docs_in
    if not docs_in or TfidfVectorizer is None:
        return None, None
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform([d['title'] + ' ' + d['text'] for d in docs_in])
    return vectorizer, X

_default_vectorizer, _default_X = build_retriever(docs)
def retrieve(query, k=2):
    if _default_vectorizer is None or _default_X is None:
        return []
    q = _default_vectorizer.transform([query])
    sims = cosine_similarity(q, _default_X)[0]
    idx = sims.argsort()[::-1][:k]
    return [{**docs[i], 'score': float(sims[i])} for i in idx]

# Shared eval helpers.
def exact_contains(output, expected):
    return str(expected).lower() in str(output).lower()

def valid_json_with_key(output, key):
    try:
        return key in json.loads(output)
    except Exception:
        return False

def run_eval(cases, system_fn):
    rows = []
    for c in cases:
        out = system_fn(c['input'])
        if 'expected' in c:
            passed = exact_contains(out, c['expected'])
        elif 'expected_key' in c:
            passed = valid_json_with_key(out, c['expected_key'])
        else:
            passed = False
        rows.append({**c, 'output': out, 'passed': passed})
    return pd.DataFrame(rows)


## Why this matters

Vision-language systems often connect a vision encoder to a language model through a learned adapter/projector. This enables image-conditioned generation or visual question answering.

This notebook builds a toy image encoder + text embedding alignment objective. It is not a full VLM, but it teaches the adapter idea.

In [ ]:
# Synthetic tiny images: vertical bars vs horizontal bars.
def make_image(label, size=8):
    img = torch.zeros(size, size)
    if label == 'vertical':
        img[:, 3:5] = 1
    else:
        img[3:5, :] = 1
    img += 0.05 * torch.randn_like(img)
    return img.clamp(0,1)

labels = ['vertical', 'horizontal'] * 16
images = torch.stack([make_image(l) for l in labels])[:, None, :, :]
text_ids = torch.tensor([0 if l=='vertical' else 1 for l in labels])
plt.figure(figsize=(3,3)); plt.imshow(images[0,0]); plt.title(labels[0]); plt.show()


In [ ]:
class TinyVisionAdapter(nn.Module):
    def __init__(self, text_dim=16):
        super().__init__()
        self.vision = nn.Sequential(
            nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(4*8*8, text_dim)
        )
        self.text = nn.Embedding(2, text_dim)
    def forward(self, img, text_id):
        img_emb = F.normalize(self.vision(img), dim=-1)
        txt_emb = F.normalize(self.text(text_id), dim=-1)
        return img_emb, txt_emb

adapter = TinyVisionAdapter()
opt = torch.optim.AdamW(adapter.parameters(), lr=0.01)
for step in range(100):
    img_emb, txt_emb = adapter(images, text_ids)
    logits = img_emb @ txt_emb.T / 0.1
    targets = torch.arange(len(images))
    # Since duplicated labels exist, this exact target is artificial; for teaching only.
    loss = F.cross_entropy(logits, targets)
    opt.zero_grad(); loss.backward(); opt.step()
print('loss:', float(loss))


In [ ]:
# Simpler classifier readout from image embedding.
with torch.no_grad():
    img_emb, _ = adapter(images, text_ids)
    class_emb = F.normalize(adapter.text(torch.tensor([0,1])), dim=-1)
    scores = img_emb @ class_emb.T
    pred = scores.argmax(-1)
    print('accuracy:', float((pred == text_ids).float().mean()))


## Production architecture sketch

```text
image → vision encoder → projector/adapter → language-model token space → LLM decoder
```

The key product questions are data quality, multimodal evals, hallucination, OCR/diagram reliability, and whether the system should answer from image evidence or use tools/RAG too.